In [0]:
# Databricks notebook source
from pyspark.sql.functions import sum, count, round

# Đọc dữ liệu từ tầng Silver
orders = spark.table("silver.orders")
items = spark.table("silver.order_items")

# 1. Tính toán doanh thu theo đơn hàng (Join Orders & Items)
revenue_df = orders.join(items, "order_id", "inner") \
    .groupBy("order_id", "purchase_ts") \
    .agg(
        round(sum("price"), 2).alias("total_revenue"),
        count("product_id").alias("item_count")
    )

spark.sql("CREATE DATABASE IF NOT EXISTS gold")
# 2. Lưu vào tầng Gold (Data Mart)
revenue_df.write.format("delta").mode("overwrite").saveAsTable("gold.daily_revenue_summary")

print("Hoàn thành tầng Gold!")